# Random Forest Classification with the Titanic Dataset

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

## Loading the Dataset

The Titanic dataset contains demographic and travel information for each passenger. The goal is to predict whether each passenger survived the disaster.

The dataset is loaded into a pandas DataFrame for exploration and preprocessing.

In [ ]:
train = pd.read_csv("titanic.csv")

train.head()

## Exploring the Dataset

Before building a model, it is important to understand the structure of the data.

The following cells display:
- dataset information
- summary statistics
- missing values

This helps determine what preprocessing steps are required before training.

In [ ]:
train.info()
train.describe()
train.isnull().sum()

## Separating Features and Labels

Machine learning models require the input features and target labels to be separated.

The target variable is **Survived**, where:

- 0 = Did not survive
- 1 = Survived

Everything else becomes an input feature.

In [ ]:
X = train.drop("Survived", axis=1)
y = train["Survived"]

## Removing Unnecessary Features

Some columns contain identifiers or free-text information that are not particularly useful for this model.

This leaves only the features that will be used for prediction.

In [ ]:
X["Deck"] = X["Cabin"].fillna("Unknown").str[0]
X = X.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

## Identifying Numerical and Categorical Features

Different types of data require different preprocessing techniques.

Numerical features can be used directly after handling missing values.

Categorical features must first be converted into numerical values using one-hot encoding.

In [ ]:
numerical_features = [
    "Age",
    "Fare",
    "SibSp",
    "Parch"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Pclass", 
    "Deck"
]

## Handling Missing Values

Machine learning models cannot train on missing values.

For numerical columns, missing values are replaced with the median of that column.

For categorical columns, missing values are replaced with the most common category.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## Encoding Categorical Variables

Machine learning algorithms operate only on numerical data.

One-hot encoding converts each category into its own binary feature.

For example:

Male → [1,0]

Female → [0,1]

This prevents the model from interpreting categories as numerical values.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

## Building the Random Forest Model

A Random Forest is an ensemble learning algorithm composed of many decision trees.

Each tree is trained on a slightly different subset of the training data and a random subset of the available features.

During prediction, every tree votes on the final class, and the majority vote becomes the model's prediction.

Using many trees generally improves accuracy while reducing overfitting compared to a single decision tree.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

## 5-Fold Cross Validation

Instead of evaluating the model using only one train-test split, cross-validation trains and evaluates the model multiple times.

For 5-fold cross-validation:

1. The dataset is divided into five equal parts.
2. Four parts are used for training.
3. One part is used for testing.
4. This process repeats five times so that every sample is tested exactly once.

The final evaluation metrics are the average across all five runs.

In [ ]:
scores = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring=[
        "accuracy",
        "precision",
        "recall"
    ]
)

## Evaluating the Model

Three evaluation metrics are computed.

**Accuracy** measures the overall percentage of correct predictions.

**Precision** measures how often passengers predicted to survive actually survived.

**Recall** measures how many actual survivors were correctly identified by the model.

Together, these metrics provide a more complete picture of model performance than accuracy alone.

In [ ]:
print(f"Accuracy : {scores['test_accuracy'].mean():.3f}")
print(f"Precision: {scores['test_precision'].mean():.3f}")
print(f"Recall   : {scores['test_recall'].mean():.3f}")